## **<font color="red">Method 01: Manual Classes</font>**

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import numpy as np

# Load data
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

# --- Linear Regression ---
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print("LR_Score:", r2_score(y_test, y_pred_lr))

# --- Ridge ---
ridge = Ridge(alpha=0.1)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
print("Ridge_Score:", r2_score(y_test, y_pred_ridge))

# --- Lasso ---
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
print("Lasso_Score:", r2_score(y_test, y_pred_lasso))

# --- ElasticNet ---
elasticnet = ElasticNet(alpha=0.005, l1_ratio=0.9)
elasticnet.fit(X_train, y_train)
y_pred_en = elasticnet.predict(X_test)
print("ElasticNet_Score:", r2_score(y_test, y_pred_en))

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- (1) Scatter plot ---
axes[0].scatter(y_test, y_pred_lr, alpha=0.7, label='Linear Regression', color='red', marker='o')
axes[0].scatter(y_test, y_pred_ridge, alpha=0.7, label='Ridge', color='blue', marker='s')
axes[0].scatter(y_test, y_pred_lasso, alpha=0.7, label='Lasso', color='green', marker='^')
axes[0].scatter(y_test, y_pred_en, alpha=0.7, label='ElasticNet', color='orange', marker='D')

# Reference line
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)

axes[0].set_xlabel("Actual Values")
axes[0].set_ylabel("Predicted Values")
axes[0].set_title("Scatter Plot of Predictions")
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

# --- (2) Lines connecting sorted points ---
sorted_idx = np.argsort(y_test)
y_test_sorted = y_test[sorted_idx]
y_pred_lr_sorted = y_pred_lr[sorted_idx]
y_pred_ridge_sorted = y_pred_ridge[sorted_idx]
y_pred_lasso_sorted = y_pred_lasso[sorted_idx]
y_pred_en_sorted = y_pred_en[sorted_idx]

axes[1].plot(y_test_sorted, y_pred_lr_sorted, color='red', marker='o', label='Linear Regression', alpha=0.8)
axes[1].plot(y_test_sorted, y_pred_ridge_sorted, color='blue', marker='s', label='Ridge', alpha=0.8)
axes[1].plot(y_test_sorted, y_pred_lasso_sorted, color='green', marker='^', label='Lasso', alpha=0.8)
axes[1].plot(y_test_sorted, y_pred_en_sorted, color='orange', marker='D', label='ElasticNet', alpha=0.8)

# Reference line
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2, label='Perfect Prediction')

axes[1].set_xlabel("Actual Values (sorted)")
axes[1].set_ylabel("Predicted Values")
axes[1].set_title("Predictions with Connecting Lines")
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

fig.tight_layout()
plt.show()


### **<font color="blue">Grid Search CV </font>**

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score

# Load data
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

# Define model
elastic = ElasticNet(max_iter=10000)

# Define parameter grid
param_grid = {
    'alpha': [0.0001, 0.001, 0.005, 0.01, 0.1, 1.0],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
}

# Grid search with cross-validation
grid_search = GridSearchCV(estimator=elastic,
                           param_grid=param_grid,
                           cv=5,           # 5-fold CV
                           scoring='r2',
                           n_jobs=-1)      # use all CPU cores

grid_search.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

# Evaluate on test data
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Test R² Score:", r2_score(y_test, y_pred))


### **<font color="blue">Random Search CV </font>**

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import r2_score

# Load data
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

# Define parameter distributions
param_dist = {
    'alpha': np.logspace(-4, 1, 100),     # 0.0001 to 10 (log scale)
    'l1_ratio': np.linspace(0, 1, 50)     # 0 to 1 evenly spaced
}

# Randomized search
random_search = RandomizedSearchCV(estimator=elastic,
                                   param_distributions=param_dist,
                                   n_iter=50,        # try 50 random combos
                                   cv=5,
                                   scoring='r2',
                                   random_state=42,
                                   n_jobs=-1)

random_search.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)

# Evaluate on test data
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Test R² Score:", r2_score(y_test, y_pred))


## **<font color="red">Method 02: SGDRegression</font>**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Load data
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

# ---------- Linear Regression (no penalty) ----------
lr = SGDRegressor(max_iter=10000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print("SGD_LinearRegression_Score:", r2_score(y_test, y_pred_lr))

# ---------- Ridge Regression (L2 penalty) ----------
ridge = SGDRegressor(penalty='l2', alpha=0.1, max_iter=10000, random_state=42)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
print("SGD_Ridge_Score:", r2_score(y_test, y_pred_ridge))

# ---------- Lasso Regression (L1 penalty) ----------
lasso = SGDRegressor(penalty='l1', alpha=0.1, max_iter=10000, random_state=42)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
print("SGD_Lasso_Score:", r2_score(y_test, y_pred_lasso))

# ---------- ElasticNet Regression (L1 + L2 penalty) ----------
elasticnet = SGDRegressor(penalty='elasticnet', alpha=0.005, l1_ratio=0.9, max_iter=10000, random_state=42)
elasticnet.fit(X_train, y_train)
y_pred_en = elasticnet.predict(X_test)
print("SGD_ElasticNet_Score:", r2_score(y_test, y_pred_en))

# --- Visualization ---
fig, axes = plt.subplots(2, 1, figsize=(10, 12))  # 2 rows, 1 column

# --- (1) Scatter plot ---
axes[0].scatter(y_test, y_pred_lr, alpha=0.7, label='Linear Regression', color='red', marker='o')
axes[0].scatter(y_test, y_pred_ridge, alpha=0.7, label='Ridge', color='blue', marker='s')
axes[0].scatter(y_test, y_pred_lasso, alpha=0.7, label='Lasso', color='green', marker='^')
axes[0].scatter(y_test, y_pred_en, alpha=0.7, label='ElasticNet', color='orange', marker='D')

# Reference line (perfect prediction)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)

axes[0].set_xlabel("Actual Values")
axes[0].set_ylabel("Predicted Values")
axes[0].set_title("Scatter Plot of Predictions")
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

# --- (2) Lines connecting sorted points ---
sorted_idx = np.argsort(y_test)
y_test_sorted = y_test[sorted_idx]
y_pred_lr_sorted = y_pred_lr[sorted_idx]
y_pred_ridge_sorted = y_pred_ridge[sorted_idx]
y_pred_lasso_sorted = y_pred_lasso[sorted_idx]
y_pred_en_sorted = y_pred_en[sorted_idx]

axes[1].plot(y_test_sorted, y_pred_lr_sorted, color='red', marker='o', label='Linear Regression', alpha=0.8)
axes[1].plot(y_test_sorted, y_pred_ridge_sorted, color='blue', marker='s', label='Ridge', alpha=0.8)
axes[1].plot(y_test_sorted, y_pred_lasso_sorted, color='green', marker='^', label='Lasso', alpha=0.8)
axes[1].plot(y_test_sorted, y_pred_en_sorted, color='orange', marker='D', label='ElasticNet', alpha=0.8)

# Reference line
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2, label='Perfect Prediction')

axes[1].set_xlabel("Actual Values (sorted)")
axes[1].set_ylabel("Predicted Values")
axes[1].set_title("Predictions with Connecting Lines")
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

fig.tight_layout()
plt.show()
